In [ ]:

import wave
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq


In [ ]:

# Step 1: Load the .wav file
file_name = "/content/zebra7.wav"  # Replace with your audio file path
wav = wave.open(file_name, 'rb')

# Extract audio parameters
num_channels = wav.getnchannels()  # Number of channels (Mono = 1, Stereo = 2)
sample_width = wav.getsampwidth()  # Bytes per sample
frame_rate = wav.getframerate()   # Sampling frequency (Hz)
num_frames = wav.getnframes()     # Total number of audio frames
duration = num_frames / frame_rate  # Total duration in seconds

# Read the audio data
frames = wav.readframes(num_frames)
wav.close()

# Step 2: Convert to numpy array based on sample width
if sample_width == 2:  # 16-bit PCM
    audio_data = np.frombuffer(frames, dtype=np.int16)
elif sample_width == 1:  # 8-bit PCM
    audio_data = np.frombuffer(frames, dtype=np.uint8) - 128  # Convert unsigned to signed
else:
    raise ValueError("Unsupported sample width. Only 8-bit or 16-bit PCM supported.")

# Step 3: If stereo, convert to mono by averaging channels
if num_channels == 2:
    audio_data = audio_data.reshape(-1, 2).mean(axis=1).astype(np.int16)

# Step 4: Normalize the audio signal to range [-1, 1]
audio_data_normalized = audio_data / np.max(np.abs(audio_data))


In [ ]:

# Step 5: Plot the full waveform to debug and visualize the entire audio
time_full = np.linspace(0, duration, num_frames, endpoint=False)
plt.figure(figsize=(12, 4))
plt.plot(time_full, audio_data_normalized, label="Normalized Audio Signal")
plt.title("Full Waveform of Audio Signal")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid()
plt.legend()
plt.show()


In [ ]:

# Step 6: Quantize the signal to 8 bits (256 levels)
audio_data_quantized = (audio_data_normalized * 127).astype(np.int8)

# Debugging: Show quantized stats
print("Quantized Audio Stats:")
print(f"Min: {audio_data_quantized.min()}, Max: {audio_data_quantized.max()}, Mean: {audio_data_quantized.mean()}")


In [ ]:

# Step 7: Extract a slice of 120 ms from a dynamic part of the audio
slice_duration = 0.120  # Duration in seconds
num_samples = int(frame_rate * slice_duration)  # Number of samples for 120 ms slice

# Extract the slice starting at 1 second or adjust as needed
slice_start = int(frame_rate * 1.0)  # Start at 1 second, you can change this value to get a different slice
if len(audio_data_quantized) >= slice_start + num_samples:
    audio_slice = audio_data_quantized[slice_start:slice_start + num_samples]
else:
    raise ValueError("Audio file is too short to extract a 120 ms slice.")

# Step 8: Plot the quantized slice
time_slice = np.linspace(0, slice_duration, num_samples, endpoint=False)
plt.figure(figsize=(10, 4))
plt.plot(time_slice, audio_slice, label="120 ms Quantized Audio Slice (8-bit)")
plt.title("Quantized Audio Slice (8-bit)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid()
plt.legend()
plt.show()


In [ ]:

# Step 1: Generate NRZ Polar Signal from Quantized Audio Slice
nrz_signal = np.where(audio_slice >= 0, 1, -1)

# Step 2: Time vector for the NRZ signal (same length as the slice)
time_nrz = np.linspace(0, slice_duration, num_samples, endpoint=False)

# Step 3: Plot the NRZ Polar Signal
plt.figure(figsize=(10, 4))
plt.step(time_nrz, nrz_signal, where='post', label="NRZ Polar Signal")
plt.title("NRZ Polar Signal Corresponding to Quantized Audio Slice")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (±1)")
plt.grid()
plt.legend()
plt.show()


In [ ]:

# Step 3: Show only the first few symbols (e.g., the first 10 symbols)
num_symbols_to_show = 100
nrz_signal_subset = nrz_signal[:num_symbols_to_show]
time_nrz_subset = time_nrz[:num_symbols_to_show]

# Step 4: Plot the NRZ Polar Signal for the first few symbols
plt.figure(figsize=(10, 4))
plt.step(time_nrz_subset, nrz_signal_subset, where='post', label="NRZ Polar Signal (First 100 Symbols)")
plt.title("First 100 Symbols of NRZ Polar Signal")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (±1)")
plt.grid()
plt.legend()
plt.show()


In [ ]:

# Step 4: Compute and plot the Discrete Signal's Power Spectrum (DSP)
# Compute the FFT of the NRZ signal
nrz_signal_fft = fft(nrz_signal)
nrz_signal_freq = fftfreq(len(nrz_signal), time_nrz[1] - time_nrz[0])

# Plot the magnitude of the FFT (Power Spectrum)
plt.figure(figsize=(10, 4))
plt.plot(nrz_signal_freq[:len(nrz_signal)//2], np.abs(nrz_signal_fft)[:len(nrz_signal)//2], label="Power Spectrum")
plt.title("Discrete Signal Power Spectrum (FFT)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.grid()
plt.legend()
plt.show()


In [ ]:

# Step 5: Plot the Eye Diagram
# Define the number of symbols per eye diagram window
symbols_per_eye = 10
eye_diagram_data = nrz_signal[:symbols_per_eye * 2]  # Get enough symbols for one eye

# Reshape the signal into segments that will form the "eyes"
eye_data = eye_diagram_data.reshape((2, symbols_per_eye))

# Plot the eye diagram
plt.figure(figsize=(10, 4))
plt.plot(np.linspace(0, slice_duration, symbols_per_eye), eye_data[0], label="Eye 1")
plt.plot(np.linspace(0, slice_duration, symbols_per_eye), eye_data[1], label="Eye 2")
plt.title("Eye Diagram of NRZ Polar Signal")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (±1)")
plt.grid()
plt.legend()
plt.show()
